[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/07-the-block-and-the-full-model.ipynb)

# Module 7: The Block and the Full Model

Companion notebook to [Module 7](https://transformers-and-math.drmhse.com/modules/07-the-block-and-the-full-model) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
torch.manual_seed(0)

## Part 1 — Sinusoidal positions and the shift property (section 7.1)

In [ ]:
def sinusoidal(n_pos, d):
    pos = torch.arange(n_pos)[:, None]
    omega = 1 / 10000 ** (torch.arange(0, d, 2) / d)
    pe = torch.zeros(n_pos, d)
    pe[:, 0::2], pe[:, 1::2] = torch.sin(pos * omega), torch.cos(pos * omega)
    return pe

pe = sinusoidal(128, 64)
plt.imshow(pe.T, aspect='auto', cmap='RdBu'); plt.xlabel('position'); plt.ylabel('feature'); plt.show()

# One 2×2 rotation per pair maps position p to p + k, whatever p is.
k, i = 7, 3
w = 1 / 10000 ** (2 * i / 64)
R = torch.tensor([[math.cos(w * k), math.sin(w * k)], [-math.sin(w * k), math.cos(w * k)]])
pair = pe[:, 2 * i:2 * i + 2]
print('max error of the shift matrix over all positions:', (pair[:-k] @ R.T - pair[k:]).abs().max().item())

## Part 2 — Residual connections give the gradient a free path (section 7.2)

In [ ]:
def grad_at_input(depth, residual):
    torch.manual_seed(0)
    layers = [nn.Sequential(nn.Linear(64, 64), nn.Tanh()) for _ in range(depth)]
    x = torch.randn(1, 64, requires_grad=True)
    h = x
    for f in layers:
        h = h + f(h) * 0.1 if residual else f(h)
    h.sum().backward()
    return x.grad.norm().item()

for depth in (4, 16, 48):
    print(f'depth {depth:2}:  plain {grad_at_input(depth, False):.2e}   residual {grad_at_input(depth, True):.2e}')

## Part 3 — LayerNorm by hand (section 7.3)

In [ ]:
x = torch.tensor([2., 4., 6., 8.])
by_hand = (x - x.mean()) / torch.sqrt(x.var(unbiased=False) + 1e-5)
print(by_hand, torch.allclose(by_hand, F.layer_norm(x, (4,)), atol=1e-5))
print('scaling the input changes nothing:', torch.allclose(F.layer_norm(x, (4,)), F.layer_norm(10 * x, (4,)), atol=1e-4))

## Part 4 — The whole model (sections 7.4–7.6)
A complete 2017-style decoder (Post-LN), with every shape printed.

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        self.attn = nn.MultiheadAttention(d, h, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
    def forward(self, x):
        N = x.shape[1]
        mask = torch.triu(torch.ones(N, N, dtype=torch.bool), 1)
        x = self.ln1(x + self.attn(x, x, x, attn_mask=mask, need_weights=False)[0])
        return self.ln2(x + self.ffn(x))

class TinyGPT(nn.Module):
    def __init__(self, vocab, d=64, h=4, L=2, ctx=128):
        super().__init__()
        self.tok, self.pos = nn.Embedding(vocab, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, h) for _ in range(L)])
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight                      # tied (section 7.5)
    def forward(self, ids, verbose=False):
        x = self.tok(ids) + self.pos(torch.arange(ids.shape[1]))
        if verbose: print('after embedding + positions', tuple(x.shape))
        for i, b in enumerate(self.blocks):
            x = b(x)
            if verbose: print(f'after block {i + 1}            ', tuple(x.shape))
        logits = self.head(x)
        if verbose: print('logits                      ', tuple(logits.shape))
        return logits

model = TinyGPT(vocab=65)
_ = model(torch.randint(0, 65, (1, 11)), verbose=True)

### Count the parameters (section 7.6)

In [ ]:
d, L, V = 64, 2, 65
formula = 12 * L * d * d + V * d
actual = sum(p.numel() for p in model.parameters())
print(f'formula 12·L·d² + |V|·d = {formula:,}   actual = {actual:,}   (the difference is biases, norms and the position table)')

## Your turn
1. Apply `F.layer_norm` to `[1, 1, 1, 5]`. What is the last entry? (Answer: about 1.732.)
2. Use the formula to estimate GPT-2 Medium's block parameters ($L = 24$, $d = 1024$).
3. Remove the `* 0.1` in Part 2. Does the residual version still pass gradient at depth 48?